In [1]:
%env UNSLOTH_RETURN_LOGITS = 1 # Run this to disable CCE since it is not supported for CPT

env: UNSLOTH_RETURN_LOGITS=1 # Run this to disable CCE since it is not supported for CPT


In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.

local_model_path = "/mnt/c/Users/rapha/hf_model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-2B",
    load_in_4bit = True,
    use_gradient_checkpointing="unsloth"
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W0519 14:36:14.264000 1520 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0519 14:36:14.454000 1520 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


<string>:1: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.


==((====))==  Unsloth 2026.5.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4050 Laptop GPU. Num GPUs = 1. Max memory: 5.997 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

In [2]:
import urllib.request
import time

url = "https://huggingface.co/unsloth/Qwen3-2B/resolve/main/model.safetensors"
start = time.time()
with urllib.request.urlopen(url, timeout=30) as r:
    chunk = r.read(1024 * 1024)  # 1MB lesen
print(f"Gelesen: {len(chunk)} bytes in {time.time()-start:.1f}s")

HTTPError: HTTP Error 401: Unauthorized

In [9]:
model.load_adapter("model_weights/lfm2_5_(1_2B)_clean")

/home/rapha/projects/copyright-backdoor/.venv/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Loading weights:   0%|          | 0/187 [00:00<?, ?it/s]

Lfm2ForCausalLM LOAD REPORT from: model_weights/lfm2_5_(1_2B)_clean
Key                                            | Status     | 
-----------------------------------------------+------------+-
base_model.model.lm_head.base_layer.weight     | UNEXPECTED | 
base_model.model.lm_head.lora_A.default.weight | UNEXPECTED | 
base_model.model.lm_head.lora_B.default.weight | UNEXPECTED | 
lm_head.lora_A.default.weight                  | MISSING    | 
lm_head.lora_B.default.weight                  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from transformers import TextIteratorStreamer
from threading import Thread
text_streamer = TextIteratorStreamer(tokenizer)
import textwrap
max_print_width = 100

# Before running inference, call `FastLanguageModel.for_inference` first

FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    "the best way "
]*1, return_tensors = "pt").to("cuda")

generation_kwargs = dict(
    inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    use_cache = True,
    do_sample = True,          # Aktiviert das kreative Sampling (wichtig!)
    temperature = 0.7,         # Kontrolliert die Zufälligkeit (0.7 ist Standard)
    top_k = 40,                # Beschränkt die Auswahl auf die Top 40 Wörter
    top_p = 0.9,               # Nucleus Sampling für bessere Qualität
    repetition_penalty = 1.2,
)

thread = Thread(target = model.generate, kwargs = generation_kwargs)
thread.start()

length = 0
for j, new_text in enumerate(text_streamer):
    if j == 0:
        wrapped_text = textwrap.wrap(new_text, width = max_print_width)
        length = len(wrapped_text[-1])
        wrapped_text = "\n".join(wrapped_text)
        print(wrapped_text, end = "")
    else:
        length += len(new_text)
        if length >= max_print_width:
            length = 0
            print()
        print(new_text, end = "")
    pass
pass

Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|startoftext|>the best way

KeyboardInterrupt: 